# EX — CNN Real-World Exercise: Image Classification (synthetic shapes)

**Goal:** build and train a small CNN to classify simple synthetic images (circles vs. squares),
so you experience the full conv -> pool -> classifier pipeline without needing a large dataset download.
Requires `torch` (and optionally `torchvision`, not required here).


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
np.random.seed(0)

def make_circle(size=28):
    img = np.zeros((size, size), dtype=np.float32)
    yy, xx = np.mgrid[0:size, 0:size]
    r = np.random.randint(6, 12)
    cx, cy = np.random.randint(r, size-r, 2)
    mask = (xx-cx)**2 + (yy-cy)**2 <= r**2
    img[mask] = 1.0
    return img

def make_square(size=28):
    img = np.zeros((size, size), dtype=np.float32)
    s = np.random.randint(8, 16)
    x0, y0 = np.random.randint(0, size-s, 2)
    img[y0:y0+s, x0:x0+s] = 1.0
    return img

n_per_class = 300
images, labels = [], []
for _ in range(n_per_class):
    images.append(make_circle()); labels.append(0)
    images.append(make_square()); labels.append(1)

images = np.stack(images)[:, None, :, :]  # (N, 1, 28, 28) -- channel dim
labels = np.array(labels)

X = torch.tensor(images)
y = torch.tensor(labels)
print(X.shape, y.shape)


In [ ]:
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)


### TODO 1
Define a small CNN: `Conv2d(1,8,3,padding=1) -> ReLU -> MaxPool2d(2) -> Conv2d(8,16,3,padding=1) -> ReLU -> MaxPool2d(2) -> Flatten -> Linear(16*7*7, 2)`.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define self.features (conv/relu/pool stack) and self.classifier (linear)
        pass

    def forward(self, x):
        # TODO
        pass

model = TinyCNN()


<details><summary>Solution</summary>

```python
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(16*7*7, 2)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

model = TinyCNN()
```
</details>


In [ ]:
model = TinyCNN() if 'TinyCNN' in dir() else None
# Guard: if TODO wasn't completed, re-define using the solution so the rest of the notebook still runs
if model is None or list(model.parameters()) == []:
    class TinyCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            )
            self.classifier = nn.Linear(16*7*7, 2)
        def forward(self, x):
            x = self.features(x)
            x = x.flatten(1)
            return self.classifier(x)
    model = TinyCNN()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    print(f"epoch {epoch}: loss={total_loss/len(X_train):.4f}")

model.eval()
with torch.no_grad():
    preds = model(X_test).argmax(dim=1)
    acc = (preds == y_test).float().mean().item()
print("test accuracy:", acc)


## Key Takeaways
- Conv layers detect local patterns; pooling downsamples and adds translation-tolerance.
- Flatten before the final `Linear` classifier head.
- Even a tiny CNN solves simple shape classification well — real datasets just need more capacity + data.
- `CrossEntropyLoss` expects raw logits (no softmax needed in the model) plus integer class labels.
